# Treccani DBI — floruit location & polity validation (sample of 10)

Builds the labeling dataset for the LLM-based verification method:

1. Sample 10 individuals with a Dizionario Biografico degli Italiani (Treccani) biography, stratified by **region of origin** × **floruit period**.
2. **Gemini 3.5 Flash, step 1** — extract the most granular floruit location, the floruit period, and word-for-word verbatim evidence (original + English). Evidence may span several extracts (`extract 1:`, `extract 2:`, …); each must be an exact substring of the biography and must explicitly contain the location name / the dates. When the text has no precise years, the dating expression as written (e.g. *prima metà del XIII secolo*) is recorded and verified instead. All checks are programmatic.
3. **Gemini 3.5 Flash, step 2** — map the location to the **smallest** Cliopatria polity governing it during the floruit (no supra-level entities); most-likely fit if the exact polity is absent; `None` only when nothing plausibly corresponds.
4. Output a TSV with empty annotation columns (single annotator).

In [1]:
import json, os, random, time, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import duckdb
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "humans_clean.duckdb").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DB_PATH = PROJECT_ROOT / "data" / "humans_clean.duckdb"
OUT_DIR = PROJECT_ROOT / "annotations" / "treccani_validation"
CACHE_DIR = OUT_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PER_BIN = 2                      # 2 individuals x 5 period bins = 10
MODEL = "google/gemini-3.5-flash"
PROMPT_VERSION = "v3"              # bump to invalidate the LLM caches
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BIO_CHAR_CAP = 25_000
PERIOD_BINS = [-10_000, 1300, 1500, 1650, 1800, 3000]
PERIOD_LABELS = ["<1300", "1300-1500", "1500-1650", "1650-1800", "1800+"]

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ["OPEN_ROUTER_API"]
random.seed(SEED)

con = duckdb.connect(str(DB_PATH), read_only=True)
print("db:", DB_PATH.name, "| model:", MODEL)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found

db: humans_clean.duckdb | model: google/gemini-3.5-flash


## 1. Candidates — DBI individuals with floruit, Italian birthplace and a Cultura polity

In [2]:
candidates = con.execute("""
    WITH dbi AS (
        SELECT wikidata_id, any_value(value) AS dbi_id
        FROM identifiers WHERE property_id = 'P1986' AND value IS NOT NULL
        GROUP BY wikidata_id
    ),
    cultura AS (
        SELECT wikidata_id, string_agg(DISTINCT polity_name, '; ') AS cultura_polities
        FROM individuals_cliopatria GROUP BY wikidata_id
    )
    SELECT i.wikidata_id, i.name_en, d.dbi_id,
           fp.floruit_year, fp.floruit_period_start, fp.floruit_period_end,
           p.name_en AS birthplace, p.lat AS birth_lat,
           c.cultura_polities
    FROM individuals i
    JOIN dbi d       ON d.wikidata_id = i.wikidata_id
    JOIN individuals_floruit_period fp ON fp.wikidata_id = i.wikidata_id
    JOIN individuals_keys k ON k.wikidata_id = i.wikidata_id
    JOIN places p    ON p.id = k.birthcity_id
                    AND p.iso_a3_code = 'ITA' AND p.lat IS NOT NULL
    JOIN cultura c   ON c.wikidata_id = i.wikidata_id
    WHERE fp.floruit_period_start IS NOT NULL
      AND fp.floruit_period_end IS NOT NULL
      AND i.non_human = 0
      AND i.name_en IS NOT NULL
""").df()

# Stable order so the seeded sample is reproducible across runs.
candidates = candidates.sort_values("wikidata_id").reset_index(drop=True)
print(f"candidates: {len(candidates):,}")
candidates.head()

candidates: 22,519


,wikidata_id,name_en,dbi_id,floruit_year,floruit_period_start,floruit_period_end,birthplace,birth_lat,cultura_polities
0,Q100076969,Luigi Caracciolo,luigi-caracciolo,<NA>,1876,1887,Andria,41.231667,Kingdom of Great Britain; British Empire
1,Q100077086,Ulisse Corticelli,ulisse-corticelli,<NA>,1850,1876,Ravenna,44.416111,Kingdom of Sardinia; Papal States; Kingdom of ...
2,Q100077443,Enrico Contessa,enrico-contessa,<NA>,1906,1932,Turin,45.079167,Kingdom of Italy
3,Q100077655,Andrea D'Angeli,andrea-d-angeli,<NA>,1897,1923,Padua,45.407778,Kingdom of Italy
4,Q100137528,Francesco Malgeri,francesco-malgeri,<NA>,1929,1955,Messina,38.193611,Kingdom of Italy; Republic of Italy


## 2. Stratified sample — region of origin × floruit period

In [3]:
# Region of origin from birthplace latitude (Italian macro-areas).
def region_of(lat):
    if lat >= 44.0:
        return "North"
    if lat >= 41.5:
        return "Center"
    return "South & Islands"

candidates["region"] = candidates["birth_lat"].apply(region_of)
candidates["period_bin"] = pd.cut(
    candidates["floruit_year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)

print(candidates.groupby(["period_bin", "region"], observed=True).size().unstack(fill_value=0))

# 2 individuals per period bin, from different regions when possible.
rows = []
for label in PERIOD_LABELS:
    pool = candidates[candidates["period_bin"] == label]
    pool = pool.sample(frac=1, random_state=SEED)
    picked = pool.drop_duplicates("region").head(N_PER_BIN)
    if len(picked) < N_PER_BIN:                      # bin with a single region
        rest = pool.drop(picked.index).head(N_PER_BIN - len(picked))
        picked = pd.concat([picked, rest])
    rows.append(picked)

sample = pd.concat(rows).reset_index(drop=True)
assert len(sample) == 10 and sample["wikidata_id"].is_unique
sample["dbi_url"] = ("https://www.treccani.it/enciclopedia/"
                     + sample["dbi_id"] + "_(Dizionario-Biografico)/")
sample[["wikidata_id", "name_en", "region", "period_bin",
        "floruit_period_start", "floruit_period_end", "cultura_polities"]]

region      Center  North  South & Islands
period_bin                                
<1300           27     17               14
1300-1500       35     28                5
1500-1650       20     35                6
1650-1800       14     13                7
1800+            9     23                3


,wikidata_id,name_en,region,period_bin,floruit_period_start,floruit_period_end,cultura_polities
0,Q1010248,Giacomino Pugliese,South & Islands,<1300,1284,1300,Kingdom of Naples
1,Q1329212,Elias of Cortona,Center,<1300,1214,1247,Papal States
2,Q41107935,Matteo Colazio,South & Islands,1300-1500,1401,1500,Kingdom of Naples; Crown of Aragon
3,Q2265857,Matteo Giovanetti,Center,1300-1500,1330,1362,Papal States
4,Q104092937,Vincenzo Luchino,North,1500-1650,1501,1600,Republic of Venice
5,Q106612626,Guarino Capello,Center,1500-1650,1526,1552,Papal States
6,Q20724583,Giuseppe Fabbrini,Center,1650-1800,1770,1802,Habsburg Monarchy; French Directory
7,Q26792385,Guglielmo Della Valle,North,1650-1800,1775,1796,Papal States
8,Q544750,Ernesto Nathan Rogers,North,1800+,1939,1969,Republic of Italy; Kingdom of Italy; Nazi Germany
9,Q327914,Indro Montanelli,Center,1800+,1935,1971,Kingdom of Italy; Republic of Italy; Nazi Germany


## 3. Fetch the DBI biographies

The article body is embedded in the page's `__NEXT_DATA__` JSON payload.

In [4]:
PAGES_PATH = CACHE_DIR / "dbi_pages.jsonl"
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}


def fetch_dbi(row):
    r = requests.get(row["dbi_url"], headers=UA, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    nd = soup.find("script", id="__NEXT_DATA__")
    content = json.loads(nd.string)["props"]["pageProps"]["data"]["content"]
    text = BeautifulSoup(content, "html.parser").get_text(" ", strip=True)
    return {"wikidata_id": row["wikidata_id"], "dbi_url": row["dbi_url"], "text": text}


pages = {}
if PAGES_PATH.exists():
    for line in PAGES_PATH.open():
        r = json.loads(line)
        pages[r["wikidata_id"]] = r

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in pages]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, PAGES_PATH.open("a") as f:
        futs = [ex.submit(fetch_dbi, row) for row in todo]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="fetch DBI"):
            r = fut.result()
            pages[r["wikidata_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

assert all(qid in pages and len(pages[qid]["text"]) > 200 for qid in sample["wikidata_id"])
pd.DataFrame([{"wikidata_id": q, "chars": len(pages[q]["text"])} for q in sample["wikidata_id"]])

,wikidata_id,chars
0,Q1010248,24603
1,Q1329212,43250
2,Q41107935,4565
3,Q2265857,23683
4,Q104092937,8085
5,Q106612626,5356
6,Q20724583,5574
7,Q26792385,17786
8,Q544750,16265
9,Q327914,42300


## 4. LLM step 1 — floruit location, floruit period, verbatim evidence

In [5]:
_tls = threading.local()


def _session():
    if not hasattr(_tls, "s"):
        s = requests.Session()
        s.headers.update({"Content-Type": "application/json",
                          "Authorization": f"Bearer {API_KEY}",
                          "HTTP-Referer": "https://bunka.ai/",
                          "X-Title": "Cultura Treccani validation"})
        _tls.s = s
    return _tls.s


def call_gemini(system, user, max_retries=4):
    body = {"model": MODEL,
            "messages": [{"role": "system", "content": system},
                         {"role": "user", "content": user}],
            "temperature": 0,
            "response_format": {"type": "json_object"}}
    last_err = ""
    for attempt in range(max_retries):
        try:
            r = _session().post(OPENROUTER_URL, json=body, timeout=180)
            if r.status_code in (408, 429, 500, 502, 503, 504):
                last_err = f"http_{r.status_code}"
                time.sleep(1.5 * 2 ** attempt)
                continue
            r.raise_for_status()
            return json.loads(r.json()["choices"][0]["message"]["content"])
        except (requests.RequestException, ValueError, KeyError) as e:
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * 2 ** attempt)
    raise RuntimeError(last_err)


STEP1_SYSTEM = """You are an expert historian reading a biography from the \
Dizionario Biografico degli Italiani (Treccani), written in Italian.

Your task: identify the ONE most granular location (a city, a region, or \
similar) associated with the individual's FLORUIT — the period during which \
they made their principal contribution — and the floruit period itself.

An individual may be linked to several locations during their active years: \
you must pick the single location best supported by the text as the place of \
their principal activity (not necessarily birthplace or deathplace).

Return STRICT JSON with exactly these keys:
  floruit_location          : most granular location name (English exonym if one exists, e.g. "Florence", "Rome")
  floruit_location_original : the location name EXACTLY as it is written in the text (e.g. "Firenze")
  floruit_location_type     : "city" | "region" | "other"
  floruit_start             : integer year (negative = BCE)
  floruit_end               : integer year
  floruit_dates_precise     : true if the text gives explicit years; false if
                              the period is expressed vaguely (e.g. a century)
  floruit_period_as_written : the dating expression EXACTLY as written in the
                              text (e.g. "prima metà del XIII secolo", "1232-1239")
  floruit_period_en         : English rendering (e.g. "first half of the 13th century")
  evidence_location_verbatim : ARRAY of verbatim extracts (original language) supporting the location
  evidence_location_en       : ARRAY of English translations, same order
  evidence_floruit_verbatim  : ARRAY of verbatim extracts supporting the floruit period
  evidence_floruit_en        : ARRAY of English translations, same order
  reasoning                 : 1-3 sentences in English

STRICT VERBATIM RULES — the extracts are the annotator's only evidence:
- Each extract MUST be copied WORD BY WORD from the text: an exact contiguous
  substring, identical spelling, accents and punctuation. No paraphrase, no
  summary, no ellipsis, no stitching of distant sentences into one extract.
- You MAY give SEVERAL extracts from different parts of the text (usually 1-3)
  when the evidence is spread out; each array element is one extract.
- At least one location extract MUST contain the location name as written in
  the text (floruit_location_original). A reader must recognise the location
  from the extracts alone.
- If floruit_dates_precise is true, at least one floruit extract MUST contain
  the year(s) on which floruit_start / floruit_end are based.
- If floruit_dates_precise is false, at least one floruit extract MUST contain
  floruit_period_as_written, and floruit_start / floruit_end are your numeric
  interpretation of it (e.g. "prima metà del XIII secolo" -> 1201-1250).
- The same extract may appear in both evidence arrays.
- Reasoning, floruit_location and floruit_period_en in English; verbatim
  extracts stay in the original language; the *_en arrays are translations."""


def step1_user(row):
    text = pages[row["wikidata_id"]]["text"][:BIO_CHAR_CAP]
    return (f"Individual: {row['name_en']} ({row['wikidata_id']})\n\n"
            f"--- DBI BIOGRAPHY ---\n{text}\n--- END ---\n\n"
            "Extract the requested fields. JSON only.")


STEP1_PATH = CACHE_DIR / f"step1_floruit_location_{PROMPT_VERSION}.jsonl"
step1 = {}
if STEP1_PATH.exists():
    for line in STEP1_PATH.open():
        r = json.loads(line)
        step1[r["wikidata_id"]] = r["extraction"]

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step1]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, STEP1_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP1_SYSTEM, step1_user(row)): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 1"):
            qid = futs[fut]
            step1[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "extraction": step1[qid]},
                               ensure_ascii=False) + "\n")

assert len(step1) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step1[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "floruit_location", "floruit_location_original",
     "floruit_location_type", "floruit_start", "floruit_end",
     "floruit_dates_precise", "floruit_period_en"]]

LLM step 1:   0%|          | 0/10 [00:00<?, ?it/s]

,wikidata_id,floruit_location,floruit_location_original,floruit_location_type,floruit_start,floruit_end,floruit_dates_precise,floruit_period_en
0,Q1010248,Sicily,Sicilia,region,1201,1250,False,first half of the 13th century
1,Q1329212,Assisi,Assisi,city,1221,1239,True,1221-1239
2,Q41107935,Padua,Padova,city,1475,1480,False,late 1470s
3,Q2265857,Avignon,Avignone,city,1322,1369,True,between 1322 and 1369
4,Q104092937,Rome,Roma,city,1551,1600,False,second half of the 16th century
5,Q106612626,Sarsina,Sarsina,city,1526,1527,True,1526-1527
6,Q20724583,Florence,Firenze,city,1771,1795,True,1771-1795
7,Q26792385,Siena,Siena,city,1780,1783,True,1780-1783
8,Q544750,Milan,Milano,city,1932,1969,True,1932-1969
9,Q327914,Milan,Milano,city,1938,1973,True,1938-1973


### 4b. Verify the verbatims are word-for-word substrings

Whitespace-normalised checks: each quote must be an exact substring of the
biography, the location quote must contain the location name as written in
the text, and the floruit quote must contain the extracted years.

In [6]:
import re


def norm(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()


def as_list(x):
    return x if isinstance(x, list) else [x]


def verify_step1(qid):
    text, e = norm(pages[qid]["text"]), step1[qid]
    loc_ex = [norm(x) for x in as_list(e["evidence_location_verbatim"])]
    flo_ex = [norm(x) for x in as_list(e["evidence_floruit_verbatim"])]
    if e.get("floruit_dates_precise"):
        years = [str(abs(int(y))) for y in (e["floruit_start"], e["floruit_end"])
                 if y is not None]
        dates_ok = any(y in q for q in flo_ex for y in years)
    else:  # vague dating: the expression as written must be quoted instead
        dates_ok = any(norm(e["floruit_period_as_written"]) in q for q in flo_ex)
    return {
        "wikidata_id": qid,
        "location_quotes_in_text": bool(loc_ex) and all(q in text for q in loc_ex),
        "floruit_quotes_in_text": bool(flo_ex) and all(q in text for q in flo_ex),
        "location_name_in_quote": any(norm(e["floruit_location_original"]) in q
                                      for q in loc_ex),
        "dates_in_quote": dates_ok,
    }


CHECK_MSGS = {
    "location_quotes_in_text": "at least one location extract is NOT an exact word-by-word substring of the biography",
    "floruit_quotes_in_text": "at least one floruit extract is NOT an exact word-by-word substring of the biography",
    "location_name_in_quote": "no location extract contains floruit_location_original",
    "dates_in_quote": "no floruit extract contains the floruit years (or floruit_period_as_written if dates are vague)",
}


def failed_checks(qid):
    v = verify_step1(qid)
    return [CHECK_MSGS[k] for k, ok in v.items() if k in CHECK_MSGS and not ok]


# One corrective retry for extractions that fail verification.
retry = [q for q in sample["wikidata_id"] if failed_checks(q)]
if retry:
    with STEP1_PATH.open("a") as f:
        for qid in tqdm(retry, desc="step 1 retry"):
            row = sample.loc[sample["wikidata_id"] == qid].iloc[0]
            feedback = ("\n\nYour previous answer failed these checks:\n- "
                        + "\n- ".join(failed_checks(qid))
                        + "\nCopy the quotes WORD BY WORD from the text and retry.")
            step1[qid] = call_gemini(STEP1_SYSTEM, step1_user(row) + feedback)
            f.write(json.dumps({"wikidata_id": qid, "extraction": step1[qid]},
                               ensure_ascii=False) + "\n")

verif = pd.DataFrame([verify_step1(q) for q in sample["wikidata_id"]])
verif["evidence_verified"] = verif.drop(columns="wikidata_id").all(axis=1)
print(f"fully verified: {int(verif['evidence_verified'].sum())}/{len(verif)}")
verif

step 1 retry:   0%|          | 0/1 [00:00<?, ?it/s]

fully verified: 10/10


,wikidata_id,location_quotes_in_text,floruit_quotes_in_text,location_name_in_quote,dates_in_quote,evidence_verified
0,Q1010248,True,True,True,True,True
1,Q1329212,True,True,True,True,True
2,Q41107935,True,True,True,True,True
3,Q2265857,True,True,True,True,True
4,Q104092937,True,True,True,True,True
5,Q106612626,True,True,True,True,True
6,Q20724583,True,True,True,True,True
7,Q26792385,True,True,True,True,True
8,Q544750,True,True,True,True,True
9,Q327914,True,True,True,True,True


## 5. LLM step 2 — map the location to a Cliopatria polity

For each individual the model chooses from the Cliopatria polities whose
period overlaps the extracted floruit.

In [7]:
def cliopatria_candidates(start, end):
    return con.execute("""
        SELECT polity_id, polity_name,
               min(from_year) AS from_year, max(to_year) AS to_year
        FROM polities_periods_cliopatria
        WHERE to_year >= ? AND from_year <= ?
        GROUP BY polity_id, polity_name
        ORDER BY polity_name
    """, [start, end]).df()


STEP2_SYSTEM = """You are an expert historical geographer.

Given a location, a floruit period, and the list of Cliopatria polities \
active during that period, choose the ONE polity that governed the location \
during the floruit period.

Return STRICT JSON with exactly these keys:
  polity_id    : integer id copied from the candidate list, or null
  polity_name  : name copied verbatim from the candidate list, or "None"
  confidence   : "high" | "medium" | "low"
  reasoning    : 1-3 sentences in English

Rules:
- polity_id and polity_name MUST come from the candidate list, no invention.
- Choose the SMALLEST (most local, most granular) polity that governed the
  location — a city-state, duchy, county or kingdom — NOT a supra-level
  entity that merely contains it (empire, personal union, alliance,
  "Minor States" aggregate, allegiance/vassalage constructs), unless no
  lower-level candidate covers the location.
- If sovereignty changed during the period, choose the polity covering the
  largest share of the period.
- If the exact governing polity is not in the list, choose the candidate
  MOST LIKELY to fit the location (geographically and politically closest),
  and lower the confidence accordingly.
- ONLY if no candidate could plausibly correspond to the location, answer
  polity_id = null and polity_name = "None", and explain in reasoning."""


def step2_user(row, ext):
    cand = cliopatria_candidates(ext["floruit_start"], ext["floruit_end"])
    lines = "\n".join(f"id={r.polity_id} | {r.polity_name} | {r.from_year} to {r.to_year}"
                      for r in cand.itertuples())
    return (f"Location: {ext['floruit_location']}\n"
            f"Floruit period: {ext['floruit_start']} to {ext['floruit_end']}\n"
            f"Individual (context only): {row['name_en']}\n\n"
            f"--- CLIOPATRIA CANDIDATE POLITIES ({len(cand)}) ---\n{lines}\n--- END ---\n\n"
            "Choose the polity. JSON only.")


STEP2_PATH = CACHE_DIR / f"step2_polity_mapping_{PROMPT_VERSION}.jsonl"


def step1_key(qid):
    e = step1[qid]
    return [e["floruit_location"], e["floruit_start"], e["floruit_end"]]


# Cache entries are only valid for the step-1 extraction they were based on.
step2 = {}
if STEP2_PATH.exists():
    for line in STEP2_PATH.open():
        r = json.loads(line)
        if r["wikidata_id"] in step1 and r.get("key") == step1_key(r["wikidata_id"]):
            step2[r["wikidata_id"]] = r["mapping"]

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step2]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, STEP2_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP2_SYSTEM,
                          step2_user(row, step1[row["wikidata_id"]])): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 2"):
            qid = futs[fut]
            step2[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "key": step1_key(qid),
                                "mapping": step2[qid]},
                               ensure_ascii=False) + "\n")

assert len(step2) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step2[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "polity_id", "polity_name", "confidence"]]

LLM step 2:   0%|          | 0/10 [00:00<?, ?it/s]

,wikidata_id,polity_id,polity_name,confidence
0,Q1010248,680.0,Kingdom of Sicily,high
1,Q1329212,419.0,Papal States,high
2,Q41107935,397.0,Republic of Venice,high
3,Q2265857,788.0,Comtat Venaissin,high
4,Q104092937,419.0,Papal States,high
5,Q106612626,419.0,Papal States,high
6,Q20724583,783.0,Holy Roman Empire Minor States,medium
7,Q26792385,NaN,None,low
8,Q544750,1428.0,Republic of Italy,high
9,Q327914,1428.0,Republic of Italy,high


## 6. Labeling TSV

One row per individual. `annot_*` columns are empty — to be filled by the
annotator (yes / no / notes). `auto_floruit_overlap_50pct` applies the rule:
the Cultura floruit is correct if it spans ≥ 50% of the extracted floruit.

In [8]:
def overlap_50pct(row):
    s, e = row["llm_floruit_start"], row["llm_floruit_end"]
    cs, ce = row["cultura_floruit_start"], row["cultura_floruit_end"]
    if pd.isna(s) or pd.isna(e) or e < s:
        return None
    ov = max(0, min(e, ce) - max(s, cs) + 1)
    return bool(ov >= 0.5 * (e - s + 1))


def fmt_extracts(x):
    return " | ".join(f"extract {i + 1}: {t}" for i, t in enumerate(as_list(x)))


records = []
for _, row in sample.iterrows():
    qid = row["wikidata_id"]
    e1, e2 = step1[qid], step2[qid]
    records.append({
        "wikidata_id": qid,
        "name": row["name_en"],
        "dbi_url": row["dbi_url"],
        "region": row["region"],
        "period_bin": str(row["period_bin"]),
        "llm_floruit_location": e1["floruit_location"],
        "llm_floruit_location_original": e1["floruit_location_original"],
        "llm_floruit_location_type": e1["floruit_location_type"],
        "llm_floruit_start": e1["floruit_start"],
        "llm_floruit_end": e1["floruit_end"],
        "llm_floruit_dates_precise": e1["floruit_dates_precise"],
        "llm_floruit_period_as_written": e1["floruit_period_as_written"],
        "llm_floruit_period_en": e1["floruit_period_en"],
        "evidence_location_verbatim": fmt_extracts(e1["evidence_location_verbatim"]),
        "evidence_location_en": fmt_extracts(e1["evidence_location_en"]),
        "evidence_floruit_verbatim": fmt_extracts(e1["evidence_floruit_verbatim"]),
        "evidence_floruit_en": fmt_extracts(e1["evidence_floruit_en"]),
        "llm_step1_reasoning": e1["reasoning"],
        "llm_polity_id": e2["polity_id"],
        "llm_polity_name": e2["polity_name"],
        "llm_polity_confidence": e2["confidence"],
        "llm_step2_reasoning": e2["reasoning"],
        "cultura_polities": row["cultura_polities"],
        "cultura_floruit_start": row["floruit_period_start"],
        "cultura_floruit_end": row["floruit_period_end"],
        "annot_location_ok": "",
        "annot_floruit_ok": "",
        "annot_polity_ok": "",
        "annot_notes": "",
    })

out = pd.DataFrame(records)
out = out.merge(verif, on="wikidata_id")
out["auto_floruit_overlap_50pct"] = out.apply(overlap_50pct, axis=1)
assert out["wikidata_id"].is_unique and len(out) == 10

OUT_PATH = OUT_DIR / "treccani_validation_sample10.tsv"
out.to_csv(OUT_PATH, sep="\t", index=False)
print("saved:", OUT_PATH)
out[["name", "region", "period_bin", "llm_floruit_location",
     "llm_floruit_start", "llm_floruit_end", "llm_polity_name",
     "cultura_polities", "evidence_verified", "auto_floruit_overlap_50pct"]]

saved: /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/annotations/treccani_validation/treccani_validation_sample10.tsv


,name,region,period_bin,llm_floruit_location,llm_floruit_start,llm_floruit_end,llm_polity_name,cultura_polities,evidence_verified,auto_floruit_overlap_50pct
0,Giacomino Pugliese,South & Islands,<1300,Sicily,1201,1250,Kingdom of Sicily,Kingdom of Naples,True,False
1,Elias of Cortona,Center,<1300,Assisi,1221,1239,Papal States,Papal States,True,True
2,Matteo Colazio,South & Islands,1300-1500,Padua,1475,1480,Republic of Venice,Kingdom of Naples; Crown of Aragon,True,True
3,Matteo Giovanetti,Center,1300-1500,Avignon,1322,1369,Comtat Venaissin,Papal States,True,True
4,Vincenzo Luchino,North,1500-1650,Rome,1551,1600,Papal States,Republic of Venice,True,True
5,Guarino Capello,Center,1500-1650,Sarsina,1526,1527,Papal States,Papal States,True,True
6,Giuseppe Fabbrini,Center,1650-1800,Florence,1771,1795,Holy Roman Empire Minor States,Habsburg Monarchy; French Directory,True,True
7,Guglielmo Della Valle,North,1650-1800,Siena,1780,1783,None,Papal States,True,True
8,Ernesto Nathan Rogers,North,1800+,Milan,1932,1969,Republic of Italy,Republic of Italy; Kingdom of Italy; Nazi Germany,True,True
9,Indro Montanelli,Center,1800+,Milan,1938,1973,Republic of Italy,Kingdom of Italy; Republic of Italy; Nazi Germany,True,True
